# Cortical Distributed Dipole Modeling via NIfTI-Based Masking

The number of dipoles is adjustable based on the spacing attribute. We then eliminate any dipole not associated with gray matter by applying a NIfTI mask. While the classic MNE-Python pipeline retains these dipoles, in our case, including non-gray matter areas interferes with our connectome construction.

## 🧪 - Experiment setup
- `ctx`: NIfTI cortical mask.
- `tkras2ras`: Spatial transform from FreeSurfer to Subject RAS.

In [1]:
# imports
import mindmap as mm
import mne
import numpy as np

# readers
dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

# data
ctx = dm.load(sub, "ctx", space="fs", segmentation="multiconn2022", scale="4")
ras2tkras = dm.load(sub, "ras2tkras")

## 🧬 - Data Processing

Build Source Spaces

In [2]:
# Setup source space
print("\033[95mSetup source space...\033[0m")
source_space = mne.setup_source_space(
    subject = f"sub-{sub}",
    spacing = "oct6",                   # dipole spacing ('octN' is a mesh downsampling method)
    surface = "white",                  # cortical mesh to use (white or pial)
    add_dist = True,                    # geodesic distances between sources
    subjects_dir = dm.config["DIR_FS"],
    n_jobs = 10,                        # number of threads
)

# mask with NIfTI
print("\033[95mMask medial wall...\033[0m")
source_space = mm.anatomy.source_space.filter_dipoles_by_nifti_mask(source_space, ctx, ras2tkras)

# summary
print("\n\033[92mSUMMARY\033[0m")
print(f"Average geodesic dipole spacing: {mm.anatomy.source_space.average_dipole_spacing(source_space, distance_type='geodesic')}")
print(f"Average euclidean dipole spacing: {mm.anatomy.source_space.average_dipole_spacing(source_space, distance_type='euclidean')}")
for src in source_space:
    print(f"Source space id: {src['id']}")
    print(f"Type: {src['type']}")
    print(f"Number of sources: {src['nuse']}\n")

Setup source space...
Setting up the source space with the following parameters:

SUBJECTS_DIR = /home/nino/Code/mindmap/data/derivatives/freesurfer
Subject      = sub-simf10
Surface      = white
Octahedron subdivision grade 6

>>> 1. Creating the source space...

Doing the octahedral vertex picking...
Loading /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-simf10/surf/lh.white...
Mapping lh sub-simf10 -> oct (6) ...
    Triangle neighbors and vertex normals...
Loading geometry from /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-simf10/surf/lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 4098/112336 selected to source space (oct = 6)

Loading /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-simf10/surf/rh.white...
Mapping rh sub-simf10 -> oct (6) ...
    Triangle neighbors and vertex normals...
Loading geometry from /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-simf10/surf/rh.sphere...
Setting up the triangulation

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   4 out of  10 | elapsed:   11.0s remaining:   16.5s
[Parallel(n_jobs=10)]: Done   7 out of  10 | elapsed:   12.3s remaining:    5.3s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:   14.3s finished
[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   4 out of  10 | elapsed:   10.3s remaining:   15.5s
[Parallel(n_jobs=10)]: Done   7 out of  10 | elapsed:   10.6s remaining:    4.6s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:   13.8s finished


    Computing patch statistics...
    Patch information added...
    Computing patch statistics...
    Patch information added...
You are now one step closer to computing the gain matrix
Mask medial wall...

SUMMARY
Average geodesic dipole spacing: 0.0037482151788474126
Average euclidean dipole spacing: 0.0031067805223921525
Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 3748

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 3717



## 📊 - Visualization

In [3]:
import numpy as np
mm.viz.plot_brain(
    source_space,
    overlay = ctx,
    ras2tkras = ras2tkras,
    mesh_params = {"show_src":True},
    overlay_params = {"color":"gray", "opacity":0.5},
).show()

Widget(value='<iframe src="http://localhost:38895/index.html?ui=P_0x769415af8290_0&reconnect=auto" class="pyvi…